## Installation

In [1]:
!pip install datasets transformers sentencepiece -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Load and Inspect TinyStories in Colab

In [5]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories", split="train")

print(ds)
print(ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2119719
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [17]:
import numpy as np

lengths = [len(x["text"].split()) for x in ds.select(range(1000))]
print("Avg words:", np.mean(lengths))
print("Max words:", np.max(lengths))

Avg words: 183.801
Max words: 837


## 2. Training our own BPE

Training our own BPE ensures:

- Tokenizer matches dataset distribution

- Model capacity is allocated efficiently

- Replication experiment remains controlled

- Small models are not handicapped by oversized vocabularies

In small-scale experiments, tokenizer choice is not minor — it materially affects model behavior.

2.1 Write texts into a plain text corpus file

In [18]:
out_path = "tiny_corpus.txt"
with open(out_path, "w", encoding="utf-8") as f:
    for ex in ds:
        txt = ex.get("text", "")
        if txt:
            # Append the <|endofstory|> token at the end of each story
            f.write(txt.replace("\r\n", "\n").replace("\r", "\n").strip() + " <|endofstory|>\n")

print("Wrote:", out_path)

Wrote: tiny_corpus.txt


In [19]:
for i in range(5):
    txt = ds[i].get("text", "")
    if txt:
        formatted_txt = txt.replace("\r\n", "\n").replace("\r", "\n").strip() + " <|endofstory|>\n"
        print(f"Story {i+1}:\n{formatted_txt}")

Story 1:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together. <|endofstory|>

Story 2:
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree ha

2.2 Train the BPE tokenizer

In [20]:
# import os
# from tokenizers import ByteLevelBPETokenizer

# tokenizer = ByteLevelBPETokenizer()
# v_size = 8000 # Adjustable

# tokenizer.train(
#     files=["tiny_corpus.txt"],
#     vocab_size=v_size,
#     min_frequency=2,
#     special_tokens=["<s>", "<pad>", "</s>", "<unk>"],
# )

# # Create the target directory in Google Drive if it doesn't exist
# save_directory = "/content/drive/MyDrive/llm_folder/"
# os.makedirs(save_directory, exist_ok=True)

# # Update file name accordingly to save in the specified drive folder
# tokenizer.save_model(save_directory)
# print(f"Saved to {save_directory}")

In [21]:
import os
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
v_size = 8000 # Adjustable

tokenizer.train(
    files=["tiny_corpus.txt"],
    vocab_size=v_size,
    min_frequency=2,
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<|endofstory|>"], # Added the new token
)

# Create the target directory in Google Drive if it doesn't exist
save_directory = "/content/drive/MyDrive/llm_folder/"
os.makedirs(save_directory, exist_ok=True)

# Save vocab.json and merges.txt
tokenizer.save_model(save_directory)
print(f"Saved tokenizer vocab.json and merges.txt to {save_directory}")

# Additionally, save the tokenizer as a single JSON file for PreTrainedTokenizerFast
tokenizer.save(os.path.join(save_directory, "tokenizer.json"))
print(f"Saved unified tokenizer.json to {save_directory}")

Saved tokenizer vocab.json and merges.txt to /content/drive/MyDrive/llm_folder/
Saved unified tokenizer.json to /content/drive/MyDrive/llm_folder/


#### 2.3 Load saved BPE tokenizer

In [22]:
import os

save_directory = "/content/drive/MyDrive/llm_folder/"

print("Files in directory:")
print(os.listdir(save_directory))

Files in directory:
['tinystories_lm_train_512', 'tinystories_lm_val_512', 'sanity_run', 'mm_coco_vit_projector_gpt256', '12layers_512emb', 'gpt2', 'models', 'lora_pythia_70m_tinystories', 'merges.txt', 'vocab.json', 'tokenizer.json']


In [23]:
from tokenizers import ByteLevelBPETokenizer
import os

save_directory = "/content/drive/MyDrive/llm_folder/"
vocab_path  = os.path.join(save_directory, "vocab.json")
merges_path = os.path.join(save_directory, "merges.txt")

bpe = ByteLevelBPETokenizer(vocab_path, merges_path)

# Save a unified tokenizer file
bpe.save(os.path.join(save_directory, "tokenizer.json"))
print("Saved tokenizer.json")

Saved tokenizer.json


In [3]:
from transformers import PreTrainedTokenizerFast
import os

save_directory = "/content/drive/MyDrive/llm_folder/"

tok = PreTrainedTokenizerFast(
    tokenizer_file=os.path.join(save_directory, "tokenizer.json"),
    bos_token="<s>",
    eos_token="<|endofstory|>", # Set <|endofstory|> as the EOS token
    unk_token="<unk>",
    pad_token="<pad>",
    additional_special_tokens=["</s>"] # Keep </s> as an additional special token
)

print("Vocab size:", tok.vocab_size)
print(tok("Once upon a time.")["input_ids"])
print(f"EOS token: {tok.eos_token}, ID: {tok.eos_token_id}")

Vocab size: 8000
[443, 459, 263, 409, 18]
EOS token: <|endofstory|>, ID: 4


#### 2.4 Decide context length

In [25]:
from datasets import load_dataset
import numpy as np

ds = load_dataset("roneneldan/TinyStories", split="train")

lens = [len(tok(t)["input_ids"]) for t in ds["text"][:2000]]
print("Avg tokens:", float(np.mean(lens)))
print("P95 tokens:", float(np.percentile(lens, 95)))
print("Max tokens:", int(np.max(lens)))

Avg tokens: 205.883
P95 tokens: 392.0
Max tokens: 980


From what observed:
- On average, each story is about 206 tokens
- 95% out of 2000 stories are shorter than 392 tokens
-> We can choose context length (block_size) of 512 for the models since it is enough to fit the whole story in

## 3. Training the model

#### 3.1 Build packed dataset

In [6]:
raw = load_dataset("roneneldan/TinyStories")
train_ds = raw["train"]
val_ds = raw["validation"] if "validation" in raw else train_ds.select(range(2000))

block_size = 512

In [7]:
def tokenize_fn(batch):
    out = tok(batch["text"], add_special_tokens=False, return_attention_mask=False)
    return out

tok_train = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

def group_texts(examples):
    concatenated = []
    for story_ids in examples["input_ids"]:
        # Extend with story tokens and then append eos_token_id if not already present
        # (it should be due to previous processing), then fill with pad_token_id.
        # This ensures that an EOS acts as a definitive end for training.
        concatenated.extend(story_ids)
        # If the last token is not EOS, or if we want to ensure separation even if EOS is already there,
        # we need a clear boundary. The simplest for fixed-size blocks is to pad after a story.
        # For this specific issue, the problem is that EOS appears *in the middle* of concatenated text.
        # A more robust approach for training EOS is to treat each story or story segment as an individual example.

    # Modify group_texts to respect EOS as an end-of-sequence signal for training
    # This approach ensures that once an EOS is encountered within a block, subsequent tokens are padded.
    input_ids = []
    labels = []
    current_chunk = []

    for story_token_list in examples["input_ids"]:
        current_chunk.extend(story_token_list)

        # Process chunks that are full or contain an EOS
        while len(current_chunk) >= block_size or (tok.eos_token_id in current_chunk and len(current_chunk) > 0):
            # Find the position of the first EOS in the current chunk
            eos_pos = -1
            if tok.eos_token_id in current_chunk:
                eos_pos = current_chunk.index(tok.eos_token_id)

            if eos_pos != -1 and eos_pos < block_size: # EOS found within the block size limit
                # Take up to and including EOS, then pad the rest of the block
                segment = current_chunk[:eos_pos + 1] # Include EOS
                current_labels = segment.copy()
                # Pad the segment to block_size
                segment.extend([tok.pad_token_id] * (block_size - len(segment)))
                current_labels.extend([tok.pad_token_id] * (block_size - len(current_labels)))
                # Shift labels for causal LM: input is token at t, label is token at t+1
                # For padding, the labels should also be padding, or -100 for ignored loss.
                # DataCollatorForLanguageModeling typically handles label shifting and -100 masking.
                # Here, we ensure the sequence ends with padding after EOS.

                input_ids.append(segment)
                labels.append(current_labels)
                current_chunk = current_chunk[eos_pos + 1:] # Remaining part for next chunk

            elif len(current_chunk) >= block_size: # No EOS, or EOS is beyond block_size
                segment = current_chunk[:block_size]
                input_ids.append(segment)
                labels.append(segment.copy())
                current_chunk = current_chunk[block_size:]
            else: # Remaining chunk is smaller than block_size and no EOS found yet
                break # Exit loop, handle this remaining part later if needed (e.g., final padding)

    # Handle any remaining chunk (less than block_size, potentially without EOS)
    if len(current_chunk) > 0:
        # If there's an EOS, pad after it
        eos_pos = -1
        if tok.eos_token_id in current_chunk:
            eos_pos = current_chunk.index(tok.eos_token_id)

        if eos_pos != -1: # EOS found in remaining chunk
            segment = current_chunk[:eos_pos + 1]
            current_labels = segment.copy()
            segment.extend([tok.pad_token_id] * (block_size - len(segment)))
            current_labels.extend([tok.pad_token_id] * (block_size - len(current_labels)))
            input_ids.append(segment)
            labels.append(current_labels)
        else: # No EOS, just a partial story chunk
            # For simplicity, we can pad this to block_size or ignore if too small
            segment = current_chunk # take the whole remaining part
            current_labels = segment.copy()
            segment.extend([tok.pad_token_id] * (block_size - len(segment)))
            current_labels.extend([tok.pad_token_id] * (block_size - len(current_labels)))
            input_ids.append(segment)
            labels.append(current_labels)

    return {"input_ids": input_ids, "labels": labels}

lm_train = tok_train.map(group_texts, batched=True, remove_columns=tok_train.column_names)
lm_val   = tok_val.map(group_texts, batched=True, remove_columns=tok_val.column_names)

print("Packed train blocks:", len(lm_train))
print("Packed val blocks:", len(lm_val))
print("One block length:", len(lm_train[0]["input_ids"]))


Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [28]:
lm_train.save_to_disk("/content/drive/MyDrive/llm_folder/tinystories_lm_train_512")
lm_val.save_to_disk("/content/drive/MyDrive/llm_folder/tinystories_lm_val_512")

Saving the dataset (0/12 shards):   0%|          | 0/909059 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9135 [00:00<?, ? examples/s]

#### 3.2 Load Packed Dataset

In [8]:
from datasets import load_from_disk

lm_train = load_from_disk("/content/drive/MyDrive/llm_folder/tinystories_lm_train_512")
lm_val   = load_from_disk("/content/drive/MyDrive/llm_folder/tinystories_lm_val_512")

print(len(lm_train), len(lm_val))

909059 9135


#### 3.3 Define a Small Sanity GPT Model (~7M)

In [30]:
import torch
from transformers import (
    GPT2Config, GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer
)

data_collator = DataCollatorForLanguageModeling(tok, mlm=False)

config = GPT2Config(
    vocab_size=8000,
    n_positions=512,
    n_ctx=512,
    n_embd=256,
    n_layer=6,
    n_head=8,
    bos_token_id=tok.bos_token_id,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id,
)

model = GPT2LMHeadModel(config)

print("Total parameters:", sum(p.numel() for p in model.parameters())/1e6, "M")

Total parameters: 6.918144 M


#### 3.4 Training Arguments

In [31]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm_folder/gpt2",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    num_train_epochs=1,
    learning_rate=5e-4,
    warmup_steps=500,
    weight_decay=0.1,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
)

# trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,3.834371,3.662008
1000,2.990255,2.842566
1500,2.641651,2.512475
2000,2.438194,2.326974
2500,2.326930,2.210942
3000,2.239491,2.130162
3500,2.191191,2.070817
4000,2.138782,2.027952
4500,2.099878,1.994555
5000,2.064122,1.964213


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=28409, training_loss=1.9949571772040315, metrics={'train_runtime': 2798.1193, 'train_samples_per_second': 324.882, 'train_steps_per_second': 10.153, 'total_flos': 1.3234471075577856e+16, 'train_loss': 1.9949571772040315, 'epoch': 1.0})

#### 3.5 Test generation

In [32]:
prompt = "Jack and Sally were running through the woods together when they stopped suddenly. They heard a strange sound coming from around the corner. “What's that?” asked Jack. Sally put her finger to her lips and said, “Shhhh. Let's go find out.” Jack and Sally followed the"
inputs = tok(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=True,
        top_p=5,
        temperature=0.9,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )

print(tok.decode(output[0], skip_special_tokens=True))

Jack and Sally were running through the woods together when they stopped suddenly. They heard a strange sound coming from around the corner. “What's that?” asked Jack. Sally put her finger to her lips and said, “Shhhh. Let's go find out.” Jack and Sally followed the noise. 
When they arrived to the cabin destination, they could see a wise old man. The old man kindly said, â€œWhy donâ€™t you pick up the horse?â€

Sally and Jack both said, â€œNo thank you for not making a grumpy man. It's important to try new things and be grateful.â€

The old old man smiled and said, â€œLetâ€™s go and discover the perfect horse together; letâ€™s go to the lake and start our adventure!â€
The old man smiled and said, â€œYes, let's go!â€

The two boys ran off in the lake, walking as fast as they could. They had seen so many animals and had a wonderful time. After the adventure paid off and they went home happy.Once there was a happy little girl. She was very shy and always told no to go to the park to do s

#### 3.6 Check n-grams

In [33]:
# from datasets import load_dataset
# import random

# def get_ngrams(tokens, n):
#     return set(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

# def ngram_containment(generated_text, tok, train_texts, n=8, max_train_docs=20000, seed=42):
#     random.seed(seed)
#     sample_texts = train_texts if len(train_texts) <= max_train_docs else random.sample(train_texts, max_train_docs)

#     # tokenize generation
#     gen_ids = tok(generated_text, add_special_tokens=False)["input_ids"]
#     if len(gen_ids) < n:
#         return {"n": n, "containment": 0.0, "gen_ngrams": 0, "train_docs_used": len(sample_texts)}
#     gen_ngrams = get_ngrams(gen_ids, n)

#     # build a set of all train ngrams (sampled docs)
#     train_ngrams = set()
#     for t in sample_texts:
#         ids = tok(t, add_special_tokens=False)["input_ids"]
#         if len(ids) >= n:
#             train_ngrams |= get_ngrams(ids, n)

#     overlap = len(gen_ngrams & train_ngrams)
#     containment = overlap / max(1, len(gen_ngrams))

#     return {
#         "n": n,
#         "containment": containment,
#         "gen_ngrams": len(gen_ngrams),
#         "train_ngrams": len(train_ngrams),
#         "train_docs_used": len(sample_texts),
#         "overlap_ngrams": overlap
#     }

# # ---- usage ----
# raw = load_dataset("roneneldan/TinyStories", split="train")
# train_texts = raw["text"]

# # generated = """PASTE YOUR GENERATED OUTPUT HERE"""
# # generated = tok.decode(output[0], skip_special_tokens=True)
# full = tok.decode(output[0], skip_special_tokens=True)
# generated = full[len(prompt):].strip()

# for n in [4, 8]:
#     stats = ngram_containment(generated, tok, train_texts, n=n, max_train_docs=20000)
#     print(stats)

#### 3.7 Save model

In [34]:
save_path = "/content/drive/MyDrive/llm_folder/models/fast"

trainer.save_model(save_path)      # saves model + config
tok.save_pretrained(save_path)     # saves tokenizer files

print("Saved to:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/llm_folder/models/fast


#### 3.8 Load model

In [9]:
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

load_path = "/content/drive/MyDrive/llm_folder/models/fast"

model2 = GPT2LMHeadModel.from_pretrained(load_path)
tok2 = PreTrainedTokenizerFast.from_pretrained(load_path)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [13]:

prompt = "Once upon a time there was a little boy named Tom. Tom loved playing outside, but he always had to be careful not to make too much noise. One day, Tom went out to explore the garden. Soon he found an unusual fruit. It looked like a prune! Tom picked up the prune and looked at it closely. It smelled sweet. Tom wanted to find out how the prune tasted, so he"
inputs = tok2(prompt, return_tensors="pt")
out = model2.generate(**inputs, do_sample=True, top_p=10, temperature=0.9, max_new_tokens=400, eos_token_id=4)
print(tok2.decode(out[0], skip_special_tokens=True))

Once upon a time there was a little boy named Tom. Tom loved playing outside, but he always had to be careful not to make too much noise. One day, Tom went out to explore the garden. Soon he found an unusual fruit. It looked like a prune! Tom picked up the prune and looked at it closely. It smelled sweet. Tom wanted to find out how the prune tasted, so he asked his mom for help.

Tom's mom was very patient. She knew where the prune could taste. She said no and told Tom that it was in the garden. She said that the prune had poison in it and it was special. Tom was very happy and he started to pick the prune.

The moral of the story is that even if something seems unusual or unknown. It might have started a bad taste. Always listen to your parents and stay safe. They think the prune may have poison spray, but it can be dangerous.One day, a girl named Jane wanted to make a sandwich. She found some bread and started to cut it with a knife. Pretty soon, she was making a face. Jane was confu